# Table Context Extraction with LightOnOCR

This notebook extracts table context from the 5 test papers in `pdfs_prueba/`.

For each evaluation table in `ground_truth_kge_evaluation.json`, it generates:
- `table_id` and `page`
- `caption` (detected after the HTML `<table>...</table>` block)
- `mentions` in narrative text (`Table N`, `Tab. N`, `Tables N and M`, etc.)

The final output is saved to:
`pdfs_prueba/ground_truth/table_context_lightonocr.json`

In [ ]:
# If needed on a clean server, uncomment these installs:
# !pip install -q torch transformers pypdfium2 pillow beautifulsoup4

import json
import re
import tempfile
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pypdfium2 as pdfium
import torch
from PIL import Image
from bs4 import BeautifulSoup
from transformers import LightOnOcrForConditionalGeneration, LightOnOcrProcessor

print("Imports OK")

In [ ]:
# Paths and run config
PDF_DIR = Path("pdfs_prueba")
GT_EVAL_PATH = Path("pdfs_prueba/ground_truth/ground_truth_kge_evaluation.json")
OUTPUT_PATH = Path("pdfs_prueba/ground_truth/table_context_lightonocr.json")
MODEL_ID = "lightonai/LightOnOCR-2-1B"
MAX_NEW_TOKENS = 4096
TARGET_LONGEST = 1540

assert PDF_DIR.exists(), f"Missing folder: {PDF_DIR}"
assert GT_EVAL_PATH.exists(), f"Missing file: {GT_EVAL_PATH}"

PDF_FILES = sorted(PDF_DIR.glob("*.pdf"))
print(f"PDFs found: {len(PDF_FILES)}")
for p in PDF_FILES:
    print(" -", p.name)

with GT_EVAL_PATH.open("r", encoding="utf-8") as f:
    gt_eval = json.load(f)

print(f"GT docs: {gt_eval.get('num_documents')} | GT eval tables: {gt_eval.get('total_tables')}")

In [ ]:
# Load LightOnOCR
if torch.cuda.is_available():
    device = "cuda"
    # bfloat16 tends to be more numerically stable than float16 on supported GPUs.
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
elif torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float16
else:
    device = "cpu"
    dtype = torch.float32

print(f"Device: {device} | dtype: {dtype}")

processor = LightOnOcrProcessor.from_pretrained(MODEL_ID)
model = LightOnOcrForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    attn_implementation="eager",
).to(device)

print("Model loaded")

In [ ]:
TABLE_REF_RE = re.compile(r"\b(?:Table|Tab\.?|TABLE|Tables|TABLES)\s+([A-Za-z]?\d+(?:\.\d+)?|[IVXLC]+)(?:\s*(?:,|and|&)\s*([A-Za-z]?\d+(?:\.\d+)?|[IVXLC]+))*", re.IGNORECASE)
# Caption start: allow 'Table N:' or 'Table N.' and optional markdown bold markers.
CAPTION_START_RE = re.compile(r"^\s*(?:\*\*)?(?:Table|Tab\.?|TABLE)\s+([A-Za-z]?\d+(?:\.\d+)?|[IVXLC]+)(?:\*\*)?\s*[:.]\s+", re.IGNORECASE)
TABLE_BLOCK_RE = re.compile(r"<table\b[^>]*>.*?</table>", flags=re.DOTALL | re.IGNORECASE)
NARRATIVE_CAPTION_VERBS_RE = re.compile(r"\b(shows?|gives?|reports?|displays?|compares?|presents?)\b", re.IGNORECASE)


def normalize_table_number(raw: str) -> str:
    return raw.strip().upper().replace(" ", "")


def normalize_table_id(raw: str) -> str:
    raw = raw.strip().lower()
    if not raw.startswith("table_"):
        return raw
    return raw


def split_sentences(text: str) -> List[str]:
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []
    parts = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", text)
    return [p.strip() for p in parts if p.strip()]


def render_pdf_page(pdf_doc, page_idx: int, target_longest: int = TARGET_LONGEST) -> Image.Image:
    page = pdf_doc[page_idx]
    bitmap = page.render(scale=200 / 72)
    img = bitmap.to_pil()

    w, h = img.size
    longest = max(w, h)
    if longest > target_longest:
        ratio = target_longest / longest
        img = img.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)

    if img.mode != "RGB":
        img = img.convert("RGB")
    return img


def ocr_page(img: Image.Image, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
    img.save(tmp, format="PNG")
    tmp.close()

    conv = [{"role": "user", "content": [{"type": "image", "url": tmp.name}]}]
    inputs = processor.apply_chat_template(
        conv,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = {
        k: v.to(device=device, dtype=dtype) if v.is_floating_point() else v.to(device)
        for k, v in inputs.items()
    }

    try:
        with torch.no_grad():
            # Deterministic decoding is more stable for OCR than sampling.
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                remove_invalid_values=True,
            )
    except RuntimeError as e:
        # Common on some GPUs: device-side assert from invalid probabilities.
        msg = str(e).lower()
        if ("device-side assert" not in msg and "cuda" not in msg) or device != "cuda":
            Path(tmp.name).unlink(missing_ok=True)
            raise

        print(" [WARN] CUDA generation failed; retrying this page on CPU/float32")
        torch.cuda.empty_cache()

        cpu_inputs = {
            k: v.to(device="cpu", dtype=torch.float32) if v.is_floating_point() else v.to("cpu")
            for k, v in inputs.items()
        }
        model_cpu = model.to("cpu")

        with torch.no_grad():
            out = model_cpu.generate(
                **cpu_inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                remove_invalid_values=True,
            )

        # Move model back for next pages.
        model_cpu.to(device)

    generated_ids = out[0, inputs["input_ids"].shape[1]:]
    result = processor.decode(generated_ids, skip_special_tokens=True)

    Path(tmp.name).unlink(missing_ok=True)
    return result


def is_weak_caption(text: str) -> bool:
    """True for weak/non-caption fragments or narrative sentences."""
    s = re.sub(r"\s+", " ", text or "").strip()
    if not s:
        return True
    if re.match(r"^(?:\*\*)?(?:Table|Tab\.?|TABLE)\s+[A-Za-z]?\d+(?:\.\d+)?\s*[:.]?\s*(?:\*\*)?$", s, flags=re.IGNORECASE):
        return True
    if not CAPTION_START_RE.search(s):
        return True
    if len(s) < 24:
        return True
    # If it reads as a narrative reference, avoid using it as caption fallback.
    if NARRATIVE_CAPTION_VERBS_RE.search(s) and "Table" in s and ":" not in s[:20]:
        return True
    return False


def normalize_caption_text(text: str, max_len: int = 420) -> str:
    """Trim caption overflow while keeping the real caption content."""
    if not text:
        return ""
    s = re.sub(r"\s+", " ", text).strip()

    # Cut at common section markers that indicate spillover beyond caption.
    cut_markers = [
        " ## ", " ### ", " 1. ", " 2. ", " 3. ", " References", " Figure ", " Fig. ",
        " Conclusion", " Conclusions", " Future Work", " Hard Negative Analysis",
    ]

    # Cut at narrative-body patterns often appended after caption in OCR.
    body_re = re.compile(
        r"\b(?:As shown|We compare|We also report|To investigate|The results|In this context|It can be observed|Our results|We evaluate)\b",
        flags=re.IGNORECASE,
    )

    cut_pos = len(s)
    for marker in cut_markers:
        pos = s.find(marker)
        if pos != -1:
            cut_pos = min(cut_pos, pos)

    m_body = body_re.search(s)
    if m_body:
        cut_pos = min(cut_pos, m_body.start())

    s = s[:cut_pos].strip(" -;,.\n\t")

    # Hard cap length to avoid paragraph drift in noisy OCR.
    if len(s) > max_len:
        s = s[:max_len].rsplit(" ", 1)[0].strip(" -;,.\n\t")

    return s


def find_caption_candidates(ocr_text: str) -> List[dict]:
    """Return caption candidates with approximate position in page text.
    Conservative strategy: keep existing strong matches and add a mild line-based fallback."""
    candidates = []

    # A) Strong sentence-level candidates (existing behavior)
    for m in CAPTION_START_RE.finditer(ocr_text):
        start = m.start()
        tail = ocr_text[start:start + 600]
        sentence = split_sentences(tail)
        if not sentence:
            continue
        s_clean = normalize_caption_text(sentence[0])
        if is_weak_caption(s_clean):
            continue
        candidates.append(
            {
                "table_num": normalize_table_number(m.group(1)),
                "caption": s_clean,
                "start": start,
            }
        )

    # B) Mild fallback for OCR line breaks: 'Table N.' + next line(s)
    lines = [ln.strip() for ln in ocr_text.splitlines()]
    line_re = re.compile(r"^(?:\*\*)?(?:Table|Tab\.?|TABLE)\s+([A-Za-z]?\d+(?:\.\d+)?|[IVXLC]+)(?:\*\*)?\s*[:.]\s*(.*)$", re.IGNORECASE)
    cursor = 0
    for i, ln in enumerate(lines):
        m = line_re.match(ln)
        if not m:
            cursor += len(ln) + 1
            continue

        num = normalize_table_number(m.group(1))
        rest = (m.group(2) or "").strip()
        merged = ln if rest else ln
        if not rest:
            extras = []
            for j in range(i + 1, min(i + 4, len(lines))):
                nxt = lines[j].strip()
                if not nxt:
                    break
                if line_re.match(nxt) or "<table" in nxt.lower():
                    break
                extras.append(nxt)
            if extras:
                merged = f"{ln} {' '.join(extras)}"

        s_clean = normalize_caption_text(merged)
        if not is_weak_caption(s_clean):
            candidates.append({"table_num": num, "caption": s_clean, "start": cursor})

        cursor += len(ln) + 1

    # Deduplicate by (table_num, caption)
    dedup = []
    seen = set()
    for c in candidates:
        key = (c["table_num"], c["caption"])
        if key in seen:
            continue
        seen.add(key)
        dedup.append(c)

    return dedup


def extract_table_blocks_with_context(ocr_text: str, page_num: int) -> List[dict]:
    """
    Extract table blocks and post-table caption candidates from one page OCR text.
    Captions are expected immediately after </table> in LightOnOCR output.
    """
    out = []
    matches = list(TABLE_BLOCK_RE.finditer(ocr_text))

    # Global candidates used as fallback when caption is not right after </table>.
    global_caption_candidates = find_caption_candidates(ocr_text)

    for i, m in enumerate(matches):
        table_html = m.group(0)
        start_after = m.end()
        end_before = matches[i + 1].start() if i + 1 < len(matches) else len(ocr_text)
        trailing = ocr_text[start_after:end_before].strip()

        caption = ""
        table_num = None

        # 1) Primary: immediate text after </table>
        if trailing:
            t_lines = [ln.strip() for ln in trailing.splitlines() if ln.strip()]
            first_line = t_lines[0] if t_lines else ""
            if CAPTION_START_RE.search(first_line):
                # Join a couple of following lines for broken OCR captions.
                caption = " ".join(t_lines[:3]).strip()
            else:
                # fallback: first sentence if it starts with Table N
                first_sentence = split_sentences(trailing[:1800])
                if first_sentence and CAPTION_START_RE.search(first_sentence[0]):
                    caption = first_sentence[0]

        if caption and not is_weak_caption(caption):
            m_cap = CAPTION_START_RE.search(caption)
            if m_cap:
                table_num = normalize_table_number(m_cap.group(1))
        else:
            caption = ""
            table_num = None

        # 2) Secondary fallback: page-level caption candidates if empty
        if not caption and global_caption_candidates:
            # Try to infer table number from nearby trailing text first (e.g. 'Table 5.')
            local_ref = TABLE_REF_RE.search(trailing[:200]) if trailing else None
            target_num = None
            if local_ref:
                nums = re.findall(r"([A-Za-z]?\d+(?:\.\d+)?|[IVXLC]+)", local_ref.group(0), flags=re.IGNORECASE)
                if nums:
                    target_num = normalize_table_number(nums[0])

            if target_num:
                for cand in global_caption_candidates:
                    if cand["table_num"] == target_num:
                        table_num, caption = cand["table_num"], cand["caption"]
                        break

            # If still empty, pick nearest caption candidate in page text.
            if not caption and len(global_caption_candidates) > 0:
                nearest = min(global_caption_candidates, key=lambda c: abs(c["start"] - start_after))
                table_num, caption = nearest["table_num"], nearest["caption"]

        caption = normalize_caption_text(caption)
        if not caption:
            table_num = None

        out.append(
            {
                "page": page_num,
                "table_html": table_html,
                "caption": caption,
                "table_num": table_num,
                "trailing_text": trailing,
            }
        )

    return out


def _clean_mention_text(text: str) -> str:
    s = re.sub(r"\s+", " ", text).strip()
    # Remove common running header artifacts.
    s = re.sub(r"^(?:Binarized Knowledge Graph Embeddings|Adversarial Contrastive Estimation|HyperKG[^\d]*|Seq2RDF[^\d]*)\s+\d+\s+", "", s, flags=re.IGNORECASE)
    return s


def extract_mentions_from_page_text(ocr_text: str, page_num: int) -> List[dict]:
    """Extract in-text mentions of Table N using paragraph-level context."""
    text_wo_tables = TABLE_BLOCK_RE.sub(" ", ocr_text)
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n+", text_wo_tables) if p.strip()]
    mentions = []

    for p in paragraphs:
        p_clean = _clean_mention_text(p)
        for m in TABLE_REF_RE.finditer(p_clean):
            matched = m.group(0)
            nums = re.findall(r"([A-Za-z]?\d+(?:\.\d+)?|[IVXLC]+)", matched, flags=re.IGNORECASE)
            norm_nums = [normalize_table_number(x) for x in nums]
            for n in norm_nums:
                mentions.append(
                    {
                        "page": page_num,
                        "table_num": n,
                        "sentence": p_clean,
                        "context": p_clean,
                    }
                )

    return mentions


def html_table_shape(table_html: str) -> Tuple[Optional[int], Optional[int]]:
    """Quick shape estimate (rows, cols) for matching against GT evaluation tables."""
    soup = BeautifulSoup(table_html, "html.parser")
    table = soup.find("table")
    if table is None:
        return None, None

    trs = table.find_all("tr")
    if not trs:
        return None, None

    row_count = max(len(trs) - 1, 0)
    first_row_cells = trs[0].find_all(["th", "td"])
    col_count = len(first_row_cells)
    return row_count, col_count


print("Helper functions loaded")

In [ ]:
def pick_pdf_for_title(paper_title: str, pdf_files: List[Path]) -> Optional[Path]:
    """Find best PDF match for GT paper title by normalized containment."""
    t = re.sub(r"[^a-z0-9]+", " ", paper_title.lower()).strip()
    for pdf in pdf_files:
        s = re.sub(r"[^a-z0-9]+", " ", pdf.stem.lower()).strip()
        if t == s or t in s or s in t:
            return pdf
    return None


def table_num_from_gt_id(table_id: str) -> Optional[str]:
    """Parse GT table_id (e.g. table_4) into normalized number for mention matching."""
    if not table_id:
        return None
    m = re.match(r"^table_([A-Za-z]?\d+(?:\.\d+)?|[IVXLC]+)$", table_id.strip(), re.IGNORECASE)
    if not m:
        return None
    return normalize_table_number(m.group(1))


def match_gt_tables_to_extracted(gt_tables: List[dict], extracted_tables: List[dict]) -> List[dict]:
    """
    Match each GT eval table with one extracted table.
    Priority: same page, then minimal shape distance.
    """
    available = list(range(len(extracted_tables)))
    matched = []

    for gt in gt_tables:
        gt_page = gt.get("page")
        gt_rows = gt.get("evaluation", {}).get("expected_rows")
        gt_cols = gt.get("evaluation", {}).get("expected_cols")

        best_idx = None
        best_score = float("inf")

        for idx in available:
            cand = extracted_tables[idx]
            page_penalty = 0 if cand["page"] == gt_page else 1000

            r, c = html_table_shape(cand["table_html"])
            if r is None or c is None or gt_rows is None or gt_cols is None:
                shape_penalty = 100
            else:
                shape_penalty = abs(r - gt_rows) + abs(c - gt_cols)

            score = page_penalty + shape_penalty
            if score < best_score:
                best_score = score
                best_idx = idx

        if best_idx is None:
            matched.append(
                {
                    "table_id": gt["table_id"],
                    "page": gt_page,
                    "caption": "",
                    "mentions": [],
                    "match_status": "not_found",
                }
            )
            continue

        cand = extracted_tables[best_idx]
        available.remove(best_idx)

        matched.append(
            {
                "table_id": gt["table_id"],
                "page": gt_page,
                "caption": cand.get("caption", ""),
                "mentions": [],
                "match_status": "matched",
            }
        )

    return matched


def extract_context_for_pdf(pdf_path: Path, gt_doc: dict) -> dict:
    print("\n" + "=" * 70)
    print(f"Processing: {pdf_path.name}")
    print("=" * 70)

    pdf_doc = pdfium.PdfDocument(str(pdf_path))
    n_pages = len(pdf_doc)

    extracted_tables = []
    all_mentions = []

    for page_idx in range(n_pages):
        page_num = page_idx + 1
        print(f"  OCR page {page_num}/{n_pages}...", end=" ", flush=True)

        img = render_pdf_page(pdf_doc, page_idx)
        ocr_text = ocr_page(img)

        page_tables = extract_table_blocks_with_context(ocr_text, page_num)
        page_mentions = extract_mentions_from_page_text(ocr_text, page_num)

        extracted_tables.extend(page_tables)
        all_mentions.extend(page_mentions)

        print(f"tables={len(page_tables)} mentions={len(page_mentions)}")

    pdf_doc.close()

    # Match only GT evaluation tables
    gt_tables = gt_doc["tables"]
    matched_tables = match_gt_tables_to_extracted(gt_tables, extracted_tables)

    # Link mentions to matched tables using GT table_id number (table_4 -> 4); fallback by page
    trivial_mention_re = re.compile(r"^\s*(?:\*\*)?(?:Table|Tab\.?|TABLE)\s+[A-Za-z]?\d+(?:\.\d+)?\s*[:.]?\s*(?:\*\*)?\s*$", re.IGNORECASE)
    caption_like_mention_re = re.compile(r"^\s*(?:\*\*)?(?:Table|Tab\.?|TABLE)\s+[A-Za-z]?\d+(?:\.\d+)?(?:\*\*)?\s*:\s*.+", re.IGNORECASE)

    for mt in matched_tables:
        ref_num = table_num_from_gt_id(mt.get("table_id", ""))
        page = mt.get("page")

        if ref_num:
            m = [x for x in all_mentions if x["table_num"] == ref_num]
        else:
            m = [x for x in all_mentions if x["page"] == page]

        # remove caption duplicates + trivial mention-only lines
        cap_norm = re.sub(r"\s+", " ", mt.get("caption", "")).strip().lower()
        unique = []
        seen = set()
        for item in m:
            s = re.sub(r"\s+", " ", item["sentence"]).strip()
            ctx = re.sub(r"\s+", " ", item.get("context", s)).strip()
            if not s:
                continue
            if len(s) < 14:
                continue
            if trivial_mention_re.match(s):
                continue
            if caption_like_mention_re.match(s):
                continue
            if cap_norm and (s.lower() == cap_norm or s.lower().startswith(cap_norm)):
                continue
            key = (item["page"], ctx)
            if key in seen:
                continue
            seen.add(key)
            unique.append({"page": item["page"], "sentence": ctx})

        mt["mentions"] = unique

    return {
        "paper_title": gt_doc["paper_title"],
        "num_tables": len(matched_tables),
        "tables": matched_tables,
    }


print("Matching + extraction pipeline loaded")

In [ ]:
# Run on the 5 GT papers and export final JSON
results_docs = []
missing = []

for gt_doc in gt_eval["documents"]:
    pdf_path = pick_pdf_for_title(gt_doc["paper_title"], PDF_FILES)
    if pdf_path is None:
        missing.append(gt_doc["paper_title"])
        print(f"[WARN] PDF not found for: {gt_doc['paper_title']}")
        continue

    doc_context = extract_context_for_pdf(pdf_path, gt_doc)
    results_docs.append(doc_context)

result_json = {
    "documents": results_docs,
    "description": "Context extraction for evaluation tables only (caption + in-text mentions), aligned with ground_truth_kge_evaluation.json",
    "num_documents": len(results_docs),
    "total_tables": sum(d["num_tables"] for d in results_docs),
    "missing_papers": missing,
}

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(result_json, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 70)
print("DONE")
print("=" * 70)
print(f"Saved: {OUTPUT_PATH}")
print(f"Documents: {result_json['num_documents']} | Tables: {result_json['total_tables']}")
if missing:
    print("Missing papers:")
    for m in missing:
        print(" -", m)

In [ ]:
# Quick preview
with OUTPUT_PATH.open("r", encoding="utf-8") as f:
    preview = json.load(f)

print(f"description: {preview['description']}")
print(f"num_documents: {preview['num_documents']} | total_tables: {preview['total_tables']}")

for doc in preview["documents"]:
    print("\n" + "-" * 60)
    print(f"Paper: {doc['paper_title']}")
    print(f"Tables: {doc['num_tables']}")
    for t in doc["tables"]:
        print(f"  {t['table_id']} (page {t['page']}) | match={t.get('match_status')} | mentions={len(t.get('mentions', []))}")
        if t.get("caption"):
            print(f"    caption: {t['caption'][:140]}...")